In [1]:
# !pip install transformers
# !pip install sacremoses
# !pip install torch
# !pip install "transformers[torch]"
# !pip install peft

# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [2]:
import sys
import os
import pandas as pd
from datasets import Dataset
import torch
from transformers import BioGptTokenizer, BioGptForCausalLM
from transformers import TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType

sys.path.append(os.path.abspath(".."))
from utils.preprocess import load_and_preprocess
from utils.llm_utils import LLMUtils


In [3]:
llmUtils = LLMUtils(("microsoft/BioGPT"))

In [4]:
# Initializing the model and tokenizer
tokenizer = BioGptTokenizer.from_pretrained("microsoft/BioGPT")
model = BioGptForCausalLM.from_pretrained("microsoft/BioGPT")

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

In [5]:
# Setting the padding end-of-sequence as padding token if padding token is not there
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

In [6]:
torch.cuda.empty_cache()
has_gpu = torch.cuda.is_available()

device = torch.device("cuda" if has_gpu else "cpu") # My lap has GPU, but making it as dynamic, so you can run with cpu as well.
model.to(device)
model.eval()

BioGptForCausalLM(
  (biogpt): BioGptModel(
    (embed_tokens): BioGptScaledWordEmbedding(42384, 1024, padding_idx=1)
    (embed_positions): BioGptLearnedPositionalEmbedding(1026, 1024)
    (layers): ModuleList(
      (0-23): 24 x BioGptDecoderLayer(
        (self_attn): BioGptAttention(
          (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
        )
        (activation_fn): GELUActivation()
        (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (fc1): Linear(in_features=1024, out_features=4096, bias=True)
        (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        (final_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      )
    )
    (layer_norm): LayerNorm((1024

In [7]:
# Load data
train_df = load_and_preprocess('../Preprocessing+baseline/data/ori_pqal.json', False)
test_df = load_and_preprocess('../Preprocessing+baseline/data/test_set.json', False)

In [8]:
# Tokenize the the training data
train_df['tokens'] = train_df.apply(llmUtils.tokenize_data, axis=1)
train_df['tokens']

0      {'input_ids': [4790, 20925, 20, 10964, 2754, 8...
1      {'input_ids': [4790, 20925, 20, 14839, 1060, 5...
2      {'input_ids': [4790, 20925, 20, 8277, 9436, 70...
3      {'input_ids': [4790, 20925, 20, 8928, 6, 281, ...
4      {'input_ids': [4790, 20925, 20, 10552, 7911, 1...
                             ...                        
995    {'input_ids': [4790, 20925, 20, 8667, 3231, 11...
996    {'input_ids': [4790, 20925, 20, 2882, 223, 420...
997    {'input_ids': [4790, 20925, 20, 2882, 1139, 27...
998    {'input_ids': [4790, 20925, 20, 10552, 4248, 1...
999    {'input_ids': [4790, 20925, 20, 8928, 9534, 9,...
Name: tokens, Length: 1000, dtype: object

In [9]:
# Create dataframe from the tokens
tokenized_df = pd.DataFrame(train_df["tokens"].tolist())
tokenized_df

,input_ids,attention_mask,labels
0,"[4790, 20925, 20, 10964, 2754, 882, 14, 151, 1...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, -100, -100, -100, -100, -100, -100, -10..."
1,"[4790, 20925, 20, 14839, 1060, 527, 103, 8, 78...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, -100, -100, -100, -100, -100, -100, -10..."
2,"[4790, 20925, 20, 8277, 9436, 70, 21220, 10, 1...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, -100, -100, -100, -100, -100, -100, -10..."
3,"[4790, 20925, 20, 8928, 6, 281, 9, 323, 62, 5,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, -100, -100, -100, -100, -100, -100, -10..."
4,"[4790, 20925, 20, 10552, 7911, 1122, 165, 1017...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, -100, -100, -100, -100, -100, -100, -10..."
...,...,...,...
995,"[4790, 20925, 20, 8667, 3231, 118, 30218, 2330...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, -100, -100, -100, -100, -100, -100, -10..."
996,"[4790, 20925, 20, 2882, 223, 420, 1505, 13, 25...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, -100, -100, -100, -100, -100, -100, -10..."
997,"[4790, 20925, 20, 2882, 1139, 276, 1003, 10, 5...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, -100, -100, -100, -100, -100, -100, -10..."
998,"[4790, 20925, 20, 10552, 4248, 1427, 9, 2019, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[-100, -100, -100, -100, -100, -100, -100, -10..."


In [10]:
# Create dataset from the tokenized dataframe
train_dataset = Dataset.from_pandas(tokenized_df[["input_ids", "attention_mask", "labels"]])
train_dataset

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1000
})

In [11]:
# Configuring LoRA to do parameter-efficient fine-tuning
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,572,864 || all params: 348,336,128 || trainable%: 0.4515


In [12]:
training_args = TrainingArguments(
    output_dir="./biogpt-lora-pubmedqa",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    learning_rate=2e-4,
    weight_decay=0.01,
    logging_steps=50,
    save_steps=500,
    save_total_limit=2,
    fp16=has_gpu,
    report_to="none",
)

In [13]:
from torch.nn.utils.rnn import pad_sequence

def custom_collate(batch):
    input_ids = [torch.tensor(x["input_ids"], dtype=torch.long) for x in batch]
    attention_mask = [torch.tensor(x["attention_mask"], dtype=torch.long) for x in batch]
    labels = [torch.tensor(x["labels"], dtype=torch.long) for x in batch]

    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_mask = pad_sequence(attention_mask, batch_first=True, padding_value=0)
    labels = pad_sequence(labels, batch_first=True, padding_value=-100)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

In [14]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=custom_collate,
)

In [15]:
trainer.train()

c:\Users\ruthr\anaconda3\Lib\site-packages\transformers\models\biogpt\modeling_biogpt.py:377: FutureWarning: `input_embeds` is deprecated and will be removed in version 5.6.0 for `create_causal_mask`. Use `inputs_embeds` instead.
  causal_mask = create_causal_mask(


Step,Training Loss
50,1.736246
100,1.285654
150,1.432240
200,1.035938
250,0.931890
300,1.036763
350,0.990121
400,1.020310
450,1.075581
500,0.825093


c:\Users\ruthr\anaconda3\Lib\site-packages\transformers\models\biogpt\modeling_biogpt.py:377: FutureWarning: `input_embeds` is deprecated and will be removed in version 5.6.0 for `create_causal_mask`. Use `inputs_embeds` instead.
  causal_mask = create_causal_mask(
c:\Users\ruthr\anaconda3\Lib\site-packages\transformers\models\biogpt\modeling_biogpt.py:377: FutureWarning: `input_embeds` is deprecated and will be removed in version 5.6.0 for `create_causal_mask`. Use `inputs_embeds` instead.
  causal_mask = create_causal_mask(


TrainOutput(global_step=1500, training_loss=0.9417849782307943, metrics={'train_runtime': 313.2154, 'train_samples_per_second': 9.578, 'train_steps_per_second': 4.789, 'total_flos': 1894896004104192.0, 'train_loss': 0.9417849782307943, 'epoch': 3.0})

In [16]:
def generate_answer(question, context, max_new_tokens=20):
    prompt = llmUtils.build_prompt(question, context)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return decoded[len(prompt):].strip()

In [17]:
predicted_values = []
reference_values = []

In [18]:
for index, row in test_df.iterrows():
    # normalizing the labels of the test data
    reference_values.append(llmUtils.normalize_label(row["label"]))

    # generating and normalizing the labels of the generated data
    generated_answer = generate_answer(row["question"], row["context"])
    predicted_values.append(llmUtils.normalize_label(generated_answer))

c:\Users\ruthr\anaconda3\Lib\site-packages\transformers\models\biogpt\modeling_biogpt.py:377: FutureWarning: `input_embeds` is deprecated and will be removed in version 5.6.0 for `create_causal_mask`. Use `inputs_embeds` instead.
  causal_mask = create_causal_mask(


In [19]:
accuracy = sum(p == r for p, r in zip(predicted_values, reference_values)) / len(reference_values)
print("Test accuracy:", accuracy)

Test accuracy: 0.654
